In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents.base import Document
from dotenv import load_dotenv
from langchain_chroma import Chroma
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]


db = Chroma.from_documents(docs, embeddings)
retriever = db.as_retriever()

In [20]:
retriever.invoke("What exactly?")

[Document(id='26639ee6-64e8-4deb-8a78-a3036ad320b7', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'),
 Document(id='ca108887-7428-4da6-944c-7c72eb4f75d0', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'),
 Document(id='f160fe92-4e03-429c-b9b4-0910791d2f5b', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'),
 Document(id='40d4ee85-2bee-4a1c-9f5b-2c0292fbec93', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza')]

In [21]:
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser


rephrase_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, 
in its original language. Keep maximum inforamtion present in standalone question.

Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

REPHRASE_TEMPLATE = PromptTemplate.from_template(rephrase_template)

model11 = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0)

rephrase_chain = REPHRASE_TEMPLATE | model11 | StrOutputParser()

rephrase_chain.invoke(
    {
        "question": "Not really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)

'Does the dog really not like to eat anything?'

In [25]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough, RunnableSequence
from operator import itemgetter

template = """Answer the question based only on the following context:
{context}   
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

c1 = (
    RunnableParallel(
        {
            "question": itemgetter("question"),
            "context": itemgetter("question") | retriever
        }
    )
    | RunnableParallel({"ques_docs" : RunnablePassthrough(), "answer": prompt | chat_model | StrOutputParser()})
)

# c1.invoke({"question": "What does the dog like to eat?"})

## Wrap rephrase_chain output (a string) into a dict for c1
#c_final = rephrase_chain | RunnableLambda(lambda x: {"question": x}) | c1

c_final = rephrase_chain | RunnableParallel({"x": RunnablePassthrough(), 
                                             "y": RunnableLambda(lambda x: {"question": x}) | c1}) 
    
c_final.invoke(
    {
        "question": "Not really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)

{'x': 'Does the dog really like to eat that?',
 'y': {'ques_docs': {'question': 'Does the dog really like to eat that?',
   'context': [Document(id='40d4ee85-2bee-4a1c-9f5b-2c0292fbec93', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
    Document(id='fc16b43e-be35-4112-b911-33b3cab4473d', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
    Document(id='0d9a3593-9053-43d6-b0c7-b989e39ad07a', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
    Document(id='26639ee6-64e8-4deb-8a78-a3036ad320b7', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]},
  'answer': 'Based on the provided information, the dog loves to eat pizza.'}}

In [26]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
ANSWER_PROMPT = ChatPromptTemplate.from_template(template)

In [27]:
from langchain_core.runnables import RunnablePassthrough

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | ANSWER_PROMPT
    | model11
    | StrOutputParser()
)

In [28]:
final_chain = rephrase_chain | retrieval_chain

In [29]:
final_chain.invoke(
    {
        "question": "Not really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)

'Based on the provided context, the dog loves to eat pizza. There is no information indicating that the dog likes to eat Thuna.'

### Chat with returning documents

In [12]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

retrieved_documents = RunnableParallel(
    {"docs": retriever, 
     "question": RunnablePassthrough()
    }
)

final_inputs = {
    "context": lambda x: "\n".join(doc.page_content for doc in x["docs"]),
    "question": lambda x: x["question"],
}
answer = {
    "answer": final_inputs | ANSWER_PROMPT | model11 | StrOutputParser(),
    "docs": lambda x: x["docs"],
}

final_chain = rephrase_chain | retrieved_documents | answer

In [13]:
result = final_chain.invoke(
    {
        "question": "Not really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)
print(result)

{'answer': 'No, the dog loves to eat pizza, so it does like to eat something.', 'docs': [Document(id='40d4ee85-2bee-4a1c-9f5b-2c0292fbec93', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'), Document(id='26639ee6-64e8-4deb-8a78-a3036ad320b7', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]}


In [14]:
result["answer"]

'No, the dog loves to eat pizza, so it does like to eat something.'

In [15]:
result["docs"]

[Document(id='40d4ee85-2bee-4a1c-9f5b-2c0292fbec93', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
 Document(id='26639ee6-64e8-4deb-8a78-a3036ad320b7', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]